# Automatic Image Captioning with CNN + LSTM — Flickr8k

**NIT3251 Group Assessment — Topic 14**

This notebook is a clean rebuild of `image_captioning.ipynb`. Every stage is commented so
that **both group members can explain every line** in the Session 10 individual Q&A, which
is worth 50% of the assessment.

## Pipeline

| Step | What happens |
|---|---|
| 1 | Load `captions.txt` with the `csv` module (handles quoted fields correctly) |
| 2 | Clean captions with **real regex** (the original used `str.replace`, a no-op) |
| 3 | Tokenise, build vocabulary |
| 4 | Seeded 80/10/10 train/val/test split, **by image** |
| 5 | Load cached VGG16 features, with a completeness guard |
| 6 | Batch generator yielding a **tuple** (Keras 3 requirement) |
| 7 | Build the encoder-decoder model |
| 8 | Train, tracking validation loss |
| 9 | Greedy decoding |
| 10 | **Beam search** with batched candidate scoring + length normalisation |
| 11 | BLEU-1..4 evaluation with smoothing |
| 12 | Beam width sweep — the headline comparison table |
| 13 | Error analysis |

## Three fixes carried over from the original notebook

1. **`data_generator` yielded a list.** `yield [X1, X2], y` must be `yield (X1, X2), y`.
   In `tf.data` a tuple denotes multiple structured inputs; a list is read as one tensor,
   so Keras cannot build an `output_signature` and `fit()` raises `TypeError`.
2. **`clean()` did nothing.** `caption.replace('[^A-Za-z]', '')` uses `str.replace`, which
   is *literal* substring replacement, not regex. It searched for the literal text
   `[^A-Za-z]`, never found it, and stripped nothing — letting 798 punctuation-contaminated
   tokens into the vocabulary.
3. **BLEU references included `startseq`/`endseq`.** The generated caption has them
   stripped, so leaving them in the references compares mismatched token streams.

## Setup

`plot_model` shells out to the Graphviz `dot` binary. Graphviz is installed system-wide but
its `bin` directory was added to the user PATH *after* VS Code started, so the kernel
inherited the old PATH. Prepending it here makes `plot_model` work without restarting.

In [ ]:
import os
import re
import csv
import time
import pickle
import random

# --- Graphviz: make `dot` findable by pydot/plot_model in this kernel -----------
# Harmless on Linux/Nix (where `dot` is already on PATH) - it just prepends a
# directory that does not exist there.
_GRAPHVIZ_BIN = r"C:\Program Files\Graphviz\bin"
if os.path.isdir(_GRAPHVIZ_BIN):
    os.environ["PATH"] = _GRAPHVIZ_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.callbacks import EarlyStopping

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import tensorflow as tf
print("TensorFlow :", tf.__version__)
print("GPUs       :", tf.config.list_physical_devices("GPU"))
# NOTE: this list is empty on native Windows. TensorFlow dropped Windows GPU
# support after 2.10, so training here is CPU-only (~3-5 min/epoch, which is fine).

## Configuration

`FAST_DEV = True` runs a tiny version of everything (1 epoch, few steps, small test set) so
you can smoke-test the whole notebook in a couple of minutes. **Set it to `False` for the
real run** that produces the numbers for the report.

In [ ]:
# ----- paths -------------------------------------------------------------------
BASE_DIR    = ""          # repo root: captions.txt and Images/ live here
WORKING_DIR = ""          # where features.pkl / model.keras are written

# ----- reproducibility ---------------------------------------------------------
# A fixed seed means the train/val/test split is identical on every run and on
# both members' machines. Without this, every re-run reshuffles the split and
# BLEU scores are not comparable between runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ----- FAST_DEV ----------------------------------------------------------------
FAST_DEV = True           # <-- set False for the real run

# ----- hyperparameters ---------------------------------------------------------
EPOCHS      = 1 if FAST_DEV else 20
BATCH_SIZE  = 32          # images per batch (not samples - see the generator)
LSTM_UNITS  = 256
EMBED_DIM   = 256
DROPOUT     = 0.4
DENSE_UNITS = 256

# ----- loss formulation --------------------------------------------------------
# True  -> integer labels + sparse_categorical_crossentropy
# False -> one-hot labels + categorical_crossentropy (what the original did)
#
# Both compute the same loss. The difference is memory: one-hot turns every single
# training sample into a vocab_size-wide float32 vector. At vocab_size ~8,765 and
# ~1,600 samples per batch that is ~54 MB of labels per batch, all of it zeros
# except one entry. Integer labels are 4 bytes each.
USE_SPARSE_LABELS = True

print(f"FAST_DEV={FAST_DEV}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}")

## Step 1 — Load captions

`captions.txt` is a CSV: `image,caption`. Some captions contain commas, and those rows are
quote-wrapped:

```
"A dog shakes its head near the shore , a red ball next to it ."
```

The original notebook split each line on `,` by hand, which breaks on those 2,447 rows — it
split the caption into fragments and left the quote characters embedded in the text. Using
`csv.reader` handles the quoting and the embedded commas correctly, so no quotes ever enter
the data and nothing needs to be stripped later.

In [ ]:
captions_path = os.path.join(BASE_DIR, "captions.txt")

# mapping: image_id (no file extension) -> list of raw caption strings
mapping = {}

with open(captions_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)                        # skip the "image,caption" header row
    for image_name, caption in reader:
        # "1000268201_693b08cb0e.jpg" -> "1000268201_693b08cb0e"
        # The feature dictionary is keyed the same way, so the two line up.
        image_id = image_name.split(".")[0]
        mapping.setdefault(image_id, []).append(caption)

n_caps = sum(len(v) for v in mapping.values())
print(f"images   : {len(mapping)}")
print(f"captions : {n_caps}")
print(f"per image: {sorted({len(v) for v in mapping.values()})}")

# Sanity check - Flickr8k is exactly 8091 images x 5 captions.
assert len(mapping) == 8091, f"expected 8091 images, got {len(mapping)}"
assert n_caps == 40455, f"expected 40455 captions, got {n_caps}"
print("\nexample:", mapping["1000268201_693b08cb0e"][0])

## Step 2 — Clean captions

### What was broken

```python
caption = caption.replace('[^A-Za-z]', '')   # literal, not regex -> does nothing
caption = caption.replace('\s+', ' ')        # literal, not regex -> does nothing
```

`str.replace` looks for an exact substring. These two lines searched for the literal text
`[^A-Za-z]` and `\s+`, never found them, and returned the string unchanged. Punctuation
survived into the vocabulary: measured over the full dataset, **798 of 9,149 vocabulary
entries were punctuation-contaminated duplicates** such as `"a` beside `a`.

`re.sub` is the regex version and actually substitutes.

### What gets removed, and why

| Characters | Count | Decision |
|---|---:|---|
| `.` | 36,599 | Remove — sentence terminator, no signal |
| `"` | 5,564 | Already gone, thanks to `csv.reader` |
| `-` | 1,083 | Remove, fusing the word: `tri-colored` -> `tricolored` |
| `'` | 437 | Remove — 400 of them are possessive `'s` |
| digits | ~260 | Remove — far too sparse to learn |
| `;:!?#&()=` | ~84 | Remove — rare noise |

Hyphens were the only real decision. Removing them fuses the compound (`tshirt`, 153
occurrences — a useful learnable token). Replacing them with a space would split it, and
`t-shirt` would degrade to just `shirt` because the single-character `t` is dropped by the
`len(word) > 1` filter.

### `startseq` / `endseq`

`startseq` seeds the decoder — it needs some input to predict the first real word from.
`endseq` is the stop signal: without it the model has no way to express "the caption is
finished" and decoding would always run to `max_length`.

In [ ]:
def clean_caption(caption):
    """Lowercase, strip non-letters, drop 1-character words, add sequence markers."""
    caption = caption.lower()
    caption = re.sub(r"[^a-z ]", "", caption)     # REAL regex (was str.replace)
    caption = re.sub(r"\s+", " ", caption).strip()
    # len(word) > 1 drops stray single characters left behind by the strip above
    # (e.g. the "s" from "dog's", the "t" from "t-shirt" if hyphens were spaced).
    words = [w for w in caption.split() if len(w) > 1]
    return "startseq " + " ".join(words) + " endseq"


# Show the fix working before applying it
demo = 'A dog shakes its head near the shore , a red ball next to it .'
print("raw    :", demo)
print("cleaned:", clean_caption(demo))

# Apply in place to every caption
for image_id, caps in mapping.items():
    for i in range(len(caps)):
        caps[i] = clean_caption(caps[i])

print("\nafter cleaning:", mapping["1000268201_693b08cb0e"][0])

## Step 3 — Tokenise and build the vocabulary

`vocab_size` is `len(word_index) + 1` because Keras reserves **index 0 for padding**.
`word_index` is 1-based, so the output layer needs one extra slot to cover index 0 even
though no real word maps to it.

`max_length` is the longest cleaned caption. It sets the input width of the sequence branch
and the maximum number of decoding steps.

In [ ]:
all_captions = [c for caps in mapping.values() for c in caps]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

vocab_size = len(tokenizer.word_index) + 1     # +1 for the padding index 0
max_length = max(len(c.split()) for c in all_captions)

# Reverse lookup, used by both decoders to turn predicted indices back into words
idx_to_word = {idx: word for word, idx in tokenizer.word_index.items()}

print(f"vocab_size : {vocab_size}")
print(f"max_length : {max_length}")

# With the clean() fix there should be no non-alphabetic tokens left at all.
dirty = [w for w in tokenizer.word_index if not w.isalpha()]
print(f"non-alphabetic tokens in vocab: {len(dirty)}  {dirty[:10]}")

# Save the tokenizer - decoding later must use the exact same word<->index map.
with open(os.path.join(WORKING_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

## Step 4 — Train / validation / test split

Two deliberate choices:

**Split by image, not by caption.** Each image has 5 captions. Splitting on captions would
put some captions of an image in train and others in test — the model would already have
seen that exact image's features, so test BLEU would be inflated. Splitting by image keeps
every image entirely on one side.

**Shuffle with a fixed seed.** `mapping.keys()` is in file order. An unshuffled split is
hard to defend as unbiased; a *seeded* shuffle is both unbiased and reproducible.

The brief implies a 90/10 train/test split. This uses **80/10/10** to carve out a
validation set, because the brief also asks for "insights into generalisation" — which
needs a validation curve measured on data the model never trains on, kept separate from the
test set used for the final BLEU numbers.

In [ ]:
image_ids = sorted(mapping.keys())          # sorted first => deterministic starting order
random.Random(SEED).shuffle(image_ids)      # seeded shuffle => reproducible split

n = len(image_ids)
n_test = n_val = int(n * 0.10)

test_keys  = image_ids[:n_test]
val_keys   = image_ids[n_test:n_test + n_val]
train_keys = image_ids[n_test + n_val:]

print(f"train : {len(train_keys)}")
print(f"val   : {len(val_keys)}")
print(f"test  : {len(test_keys)}")

# No image may appear in two splits.
assert not (set(train_keys) & set(val_keys))
assert not (set(train_keys) & set(test_keys))
assert not (set(val_keys) & set(test_keys))
print("\nsplits are disjoint")

## Step 5 — Load cached VGG16 features

`features.pkl` maps `image_id -> (1, 4096)` array: the activations of VGG16's `fc2` layer,
its second-to-last layer.

**Why `fc2` and not the final layer?** The final layer outputs 1,000 ImageNet *class
scores* — a prediction about which of 1,000 categories the image belongs to. `fc2` is the
4,096-dimensional general-purpose representation feeding that classifier, which is far more
informative for describing an image.

**Why cache?** Extraction costs ~183 ms/image, so ~25 minutes for all 8,091. VGG16 is
frozen, so its output for a given image never changes — recomputing it every epoch would
waste that 25 minutes 20 times over.

**What it costs:** flattening to a single 4,096-d vector discards all spatial information.
The model knows *what* is in the image but not *where*. This is the direct motivation for
attention as future work.

The guard below matters: a previous run of the original notebook was interrupted and pickled
a partial dictionary of only 111 images, which fails later as a confusing `KeyError` deep
inside the generator.

In [ ]:
features_path = os.path.join(WORKING_DIR, "features.pkl")
with open(features_path, "rb") as f:
    features = pickle.load(f)

print(f"features   : {len(features)} images")
print(f"shape      : {features[next(iter(features))].shape}")

# GUARD: fail loudly and immediately if extraction was incomplete.
missing = [k for k in mapping if k not in features]
assert not missing, (
    f"features.pkl is incomplete: {len(missing)} of {len(mapping)} images missing "
    f"(first: {missing[0]}). Re-run feature extraction before training."
)
print("every image has a cached feature")

## Step 6 — Batch generator

### How one caption becomes many training samples

The model predicts **one word at a time**, given the image and the words so far. So the
caption `startseq a dog runs endseq` becomes four training samples:

| Image feature | Input sequence | Target word |
|---|---|---|
| `f` | `startseq` | `a` |
| `f` | `startseq a` | `dog` |
| `f` | `startseq a dog` | `runs` |
| `f` | `startseq a dog runs` | `endseq` |

This is **teacher forcing**: the input prefix always comes from the ground-truth caption,
not from what the model previously predicted. It makes training stable and parallel, but it
is also why decoding is harder than training — at decode time the model must feed on its
own output, and there is no ground truth to fall back on.

With ~6,500 training images × 5 captions × ~11 words, one epoch is roughly 350,000 samples.
That will not fit in memory as one array, which is why this is a generator.

### The Keras 3 fix

`yield (X1, X2), y` — a **tuple**, not a list. In `tf.data`, a tuple means "several
structured inputs" while a list is interpreted as a single tensor. Yielding a list makes
Keras try to build one `TypeSpec` from two differently shaped arrays, which fails with:

```
TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec`
           but found <class 'list'> which is not.
```

Note `BATCH_SIZE` counts **images**, not samples — each image contributes ~55 samples, so a
batch of 32 images is ~1,760 training samples.

In [ ]:
def data_generator(keys, mapping, features, tokenizer, max_length,
                   vocab_size, batch_size, sparse_labels=True):
    """Yield ((image_features, input_sequences), targets) forever."""
    X1, X2, y = [], [], []
    n_images = 0
    while True:                                  # Keras needs an endless generator
        for key in keys:
            n_images += 1
            for caption in mapping[key]:
                seq = tokenizer.texts_to_sequences([caption])[0]
                # Expand one caption into (prefix -> next word) pairs
                for i in range(1, len(seq)):
                    in_seq, out_word = seq[:i], seq[i]
                    # Pre-pad: zeros go on the LEFT, so real tokens sit at the end
                    # adjacent to the prediction point. Pairs with mask_zero below.
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    X1.append(features[key][0])
                    X2.append(in_seq)
                    if sparse_labels:
                        y.append(out_word)                       # just the index
                    else:
                        y.append(to_categorical([out_word],
                                                num_classes=vocab_size)[0])
            if n_images == batch_size:
                # THE FIX: a tuple for the inputs, not a list.
                yield (np.array(X1), np.array(X2)), np.array(y)
                # Reset the buffers, otherwise leftovers bleed into the next batch
                X1, X2, y = [], [], []
                n_images = 0


# Peek at one batch to confirm the shapes
_peek = data_generator(train_keys, mapping, features, tokenizer, max_length,
                       vocab_size, BATCH_SIZE, USE_SPARSE_LABELS)
(_x1, _x2), _y = next(_peek)
print(f"image features : {_x1.shape}   (samples, 4096)")
print(f"input sequences: {_x2.shape}   (samples, max_length)")
print(f"targets        : {_y.shape}   {'integer labels' if USE_SPARSE_LABELS else 'one-hot'}")
print(f"\n~{_x1.shape[0]} samples per batch of {BATCH_SIZE} images")
print(f"label memory   : {_y.nbytes / 1e6:.2f} MB per batch")
del _peek, _x1, _x2, _y

## Step 7 — Model architecture

Two input branches merged into one decoder:

```
image (4096,)  -> Dropout -> Dense(256, relu) ---+
                                                 |-> add -> Dense(256, relu) -> Dense(vocab, softmax)
caption (35,)  -> Embedding -> Dropout -> LSTM(256) +
```

**Why `add` rather than `concatenate`?** `add` requires both branches to be the same width
(256) and sums them into a 256-d vector. `concatenate` would produce 512-d and more
decoder parameters. Addition forces the two modalities into a *shared* 256-d space, which
is the standard formulation for this architecture and keeps the model small — worth knowing
as a design decision you chose, not inherited.

**Why `mask_zero=True`?** Sequences are pre-padded with zeros. Without masking, the LSTM
would process those zeros as real tokens. `mask_zero=True` tells downstream layers to skip
them, so a 3-word prefix is treated as 3 words rather than 32 pad tokens plus 3 words.

**Why dropout on both branches?** 6,500 images is a small dataset for ~6M parameters.
Dropout at 0.4 is aggressive regularisation against memorising captions.

In [ ]:
def build_model(vocab_size, max_length, lstm_units=LSTM_UNITS, embed_dim=EMBED_DIM,
                dropout=DROPOUT, dense_units=DENSE_UNITS, sparse_labels=True):
    """Build and compile the encoder-decoder captioning model."""
    # --- image branch (encoder output is already computed by VGG16) ---
    inputs1 = Input(shape=(4096,), name="image_features")
    fe1 = Dropout(dropout)(inputs1)
    fe2 = Dense(dense_units, activation="relu")(fe1)

    # --- sequence branch (the caption so far) ---
    inputs2 = Input(shape=(max_length,), name="caption_sequence")
    se1 = Embedding(vocab_size, embed_dim, mask_zero=True)(inputs2)
    se2 = Dropout(dropout)(se1)
    se3 = LSTM(lstm_units)(se2)        # returns only the final hidden state

    # --- decoder: merge the two branches and predict the next word ---
    decoder1 = add([fe2, se3])         # element-wise sum; both are `dense_units` wide
    decoder2 = Dense(dense_units, activation="relu")(decoder1)
    outputs = Dense(vocab_size, activation="softmax")(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    # sparse_* takes integer targets; the non-sparse version takes one-hot.
    # Mathematically identical - see USE_SPARSE_LABELS in the config cell.
    loss = "sparse_categorical_crossentropy" if sparse_labels else "categorical_crossentropy"
    model.compile(loss=loss, optimizer="adam")
    return model


model = build_model(vocab_size, max_length, sparse_labels=USE_SPARSE_LABELS)
model.summary()

In [ ]:
# Architecture diagram - also a good figure for the report.
# Requires the Graphviz `dot` binary (see the setup cell).
plot_model(model, to_file=os.path.join(WORKING_DIR, "model.png"),
           show_shapes=True, show_layer_names=True)
print("wrote model.png")

## Step 8 — Train

`steps_per_epoch` must be set explicitly because the generator is endless — it is how Keras
knows where one epoch ends.

Validation loss is what tells you about **generalisation**. Training loss falling while
validation loss rises is overfitting, and the epoch where validation loss bottoms out is
the model worth keeping. `EarlyStopping` with `restore_best_weights=True` does that
automatically.

Measured on CPU: ~0.8 s/step, ~200 steps/epoch, so **~3 min/epoch** and roughly 1 hour for
20 epochs.

In [ ]:
steps_per_epoch  = len(train_keys) // BATCH_SIZE
validation_steps = max(1, len(val_keys) // BATCH_SIZE)

if FAST_DEV:                      # tiny run just to prove the pipeline works
    steps_per_epoch, validation_steps = 10, 2

print(f"steps/epoch={steps_per_epoch}  val steps={validation_steps}  epochs={EPOCHS}")

train_gen = data_generator(train_keys, mapping, features, tokenizer, max_length,
                           vocab_size, BATCH_SIZE, USE_SPARSE_LABELS)
val_gen   = data_generator(val_keys, mapping, features, tokenizer, max_length,
                           vocab_size, BATCH_SIZE, USE_SPARSE_LABELS)

early = EarlyStopping(monitor="val_loss", patience=3,
                      restore_best_weights=True, verbose=1)

t0 = time.time()
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    callbacks=[early],
    verbose=1,
)
print(f"\ntrained in {(time.time() - t0) / 60:.1f} min")

model_path = os.path.join(WORKING_DIR, "model.keras")
model.save(model_path)
print("saved", model_path)

In [ ]:
# Loss curves - a required figure for the report.
hist = history.history
plt.figure(figsize=(7, 4))
plt.plot(hist["loss"], marker="o", label="train loss")
if "val_loss" in hist:
    plt.plot(hist["val_loss"], marker="s", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "loss_curve.png"), dpi=150)
plt.show()

# Also persist the raw numbers so the report can quote them without re-training.
with open(os.path.join(WORKING_DIR, "history.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["epoch"] + list(hist.keys()))
    for i in range(len(hist["loss"])):
        w.writerow([i + 1] + [hist[k][i] for k in hist])
print("wrote history.csv")

## Step 9 — Greedy decoding

The simplest decoder: at each step take the single highest-probability word
(`argmax`), append it, repeat until `endseq` or `max_length`.

Greedy is **locally** optimal and **globally** short-sighted. Once it commits to a word it
can never revisit that choice, even if it leads into a dead end. A slightly less likely
first word might open a much better caption overall — which is exactly what beam search
exists to explore.

In [ ]:
def greedy_decode(model, feature, tokenizer, idx_to_word, max_length):
    """Decode one caption by taking the argmax word at every step."""
    in_text = "startseq"
    for _ in range(max_length):
        seq = tokenizer.texts_to_sequences([in_text])[0]
        seq = pad_sequences([seq], maxlen=max_length)
        yhat = model.predict([feature, seq], verbose=0)
        word = idx_to_word.get(int(np.argmax(yhat[0])))
        if word is None:          # index 0 (padding) - no word maps to it
            break
        in_text += " " + word
        if word == "endseq":      # model signalled the end of the caption
            break
    # Strip the markers; they are scaffolding, not part of the caption
    return " ".join(w for w in in_text.split() if w not in ("startseq", "endseq"))


_k = test_keys[0]
print("greedy :", greedy_decode(model, features[_k], tokenizer, idx_to_word, max_length))
print("\nreferences:")
for c in mapping[_k]:
    print("  ", " ".join(w for w in c.split() if w not in ("startseq", "endseq")))

## Step 10 — Beam search (the extended challenge)

Beam search keeps the **k best partial captions** at every step instead of just one. At
each timestep it expands all k beams, scores every continuation, and keeps the k best
overall. It explores more of the search space, so it can recover from a locally attractive
but globally poor first word.

### Two implementation details that matter

**1. Batch the candidate scoring.** Measured on this machine:

| `model.predict` call | Time |
|---|---|
| batch of 1 | 57 ms |
| batch of 10 | 58 ms |

Per-call overhead dominates completely — 10× the work is free. Scoring each beam in its own
`predict` call makes BLEU evaluation take ~46 minutes; batching all k beams into **one call
per timestep** makes it ~12 minutes. This loop does the latter.

**2. Length normalisation.** Scores are sums of log-probabilities, and every
log-probability is negative. So a longer caption always scores worse simply for being
longer, and raw beam search systematically prefers short, generic captions. Dividing the
total by the token count removes that bias. The flag below lets you report both, which is a
genuinely interesting comparison rather than a footnote.

In [ ]:
def beam_search_decode(model, feature, tokenizer, idx_to_word, max_length,
                       beam_width=5, length_norm=True):
    """Decode one caption with beam search, scoring all beams in one predict call."""
    start_id = tokenizer.word_index["startseq"]
    end_id   = tokenizer.word_index["endseq"]

    live = [([start_id], 0.0)]      # (token_ids, cumulative log-probability)
    done = []                       # beams that have emitted endseq

    for _ in range(max_length):
        if not live:
            break

        # --- score every live beam in ONE predict call (the 4x speedup) ---
        seqs  = pad_sequences([toks for toks, _ in live], maxlen=max_length)
        feats = np.repeat(feature, len(live), axis=0)      # same image, k times
        probs = model.predict([feats, seqs], verbose=0)    # (k, vocab_size)
        logp  = np.log(probs + 1e-12)                      # epsilon avoids log(0)

        # --- expand each beam by its top-k continuations ---
        candidates = []
        for i, (toks, score) in enumerate(live):
            for tok in np.argsort(logp[i])[-beam_width:]:
                candidates.append((toks + [int(tok)], score + float(logp[i][tok])))

        # --- keep the k best overall, retiring any that finished ---
        candidates.sort(key=lambda c: c[1], reverse=True)
        live = []
        for toks, score in candidates[:beam_width]:
            if toks[-1] == end_id:
                done.append((toks, score))
            else:
                live.append((toks, score))

        if len(done) >= beam_width:   # enough finished captions to choose from
            break

    done.extend(live)                 # include anything still running at max_length

    # Length normalisation: divide by token count, otherwise short captions win
    def final_score(beam):
        toks, score = beam
        return score / len(toks) if length_norm else score

    best_tokens = max(done, key=final_score)[0]
    words = [idx_to_word.get(t) for t in best_tokens]
    return " ".join(w for w in words if w and w not in ("startseq", "endseq"))


for k in (1, 3, 5):
    print(f"beam k={k}: {beam_search_decode(model, features[_k], tokenizer, idx_to_word, max_length, k)}")

### Sanity check: beam width 1 must equal greedy

With `k=1` beam search keeps exactly one hypothesis and takes the single best continuation
at every step — which *is* greedy decoding. If these two disagree, the beam implementation
has a bug. This is a cheap, decisive correctness test, and worth stating in the report.

(Length normalisation is irrelevant at `k=1`: dividing a single candidate's score by its
length cannot change which candidate is selected when there is only one.)

In [ ]:
mismatches = []
for key in test_keys[:15]:
    g = greedy_decode(model, features[key], tokenizer, idx_to_word, max_length)
    b = beam_search_decode(model, features[key], tokenizer, idx_to_word, max_length,
                           beam_width=1)
    if g != b:
        mismatches.append((key, g, b))

if mismatches:
    print(f"MISMATCH on {len(mismatches)}/15 images - beam search has a bug")
    for key, g, b in mismatches[:3]:
        print(f"  {key}\n    greedy: {g}\n    beam-1: {b}")
else:
    print("PASS - beam width 1 reproduces greedy on all 15 images")

## Step 11 — BLEU evaluation

BLEU compares generated n-grams against the 5 human reference captions. BLEU-1 measures
single words (roughly: did you name the right things?); BLEU-4 measures 4-word sequences
(roughly: is the phrasing right?). BLEU-4 is much harsher and the usual headline number.

**Two correctness details:**

**Strip the markers from the references.** The generated caption has `startseq`/`endseq`
removed, so leaving them in the references compares mismatched token streams and distorts
the score. The original notebook left them in.

**Use a smoothing function.** BLEU-4 needs matching 4-grams. On short captions a single
hypothesis often has *zero* matching 4-grams, and the geometric mean then collapses the
whole score to 0. Smoothing avoids reporting a misleading 0.0.

**BLEU's weakness, worth stating in the report:** it is pure n-gram overlap, so a correct
paraphrase using different words scores poorly. "A dog runs through grass" versus "A canine
sprints across a lawn" means the same thing and scores near zero. CIDEr, METEOR and SPICE
were designed to address this.

In [ ]:
def strip_markers(caption):
    return [w for w in caption.split() if w not in ("startseq", "endseq")]


def evaluate(model, keys, mapping, features, tokenizer, idx_to_word, max_length,
             decoder="greedy", beam_width=5, length_norm=True, show_progress=True):
    """Generate captions for `keys` and return BLEU-1..4 plus the raw predictions."""
    references, hypotheses = [], []
    t0 = time.time()

    for key in (tqdm(keys, desc=decoder) if show_progress else keys):
        if decoder == "greedy":
            pred = greedy_decode(model, features[key], tokenizer, idx_to_word, max_length)
        else:
            pred = beam_search_decode(model, features[key], tokenizer, idx_to_word,
                                      max_length, beam_width, length_norm)
        # corpus_bleu expects: references = list of lists of token-lists
        #                      hypotheses = list of token-lists
        references.append([strip_markers(c) for c in mapping[key]])
        hypotheses.append(pred.split())

    smooth = SmoothingFunction().method1
    bleu = {
        f"BLEU-{n}": corpus_bleu(references, hypotheses,
                                 weights=tuple([1 / n] * n),
                                 smoothing_function=smooth)
        for n in (1, 2, 3, 4)
    }
    bleu["seconds"] = time.time() - t0
    return bleu, hypotheses


# FAST_DEV keeps this to a handful of images; the real run uses all ~809.
eval_keys = test_keys[:25] if FAST_DEV else test_keys
print(f"evaluating on {len(eval_keys)} test images")

greedy_bleu, greedy_preds = evaluate(model, eval_keys, mapping, features, tokenizer,
                                     idx_to_word, max_length, decoder="greedy")
for k, v in greedy_bleu.items():
    print(f"  {k}: {v:.4f}")

## Step 12 — Decoder comparison (the headline table)

This is the table the report is built around. It directly answers the assessment's
"comparison of decoding strategies" deliverable.

Expect beam search to beat greedy modestly, then **plateau or regress** at larger widths.
If that happens, report it and explain it: a wider beam finds higher-probability captions,
and the highest-probability caption is usually a safe generic one ("a man in a red shirt"),
which overlaps less with specific human references. An honest negative result analysed
properly is worth more marks than a suspiciously clean win.

In [ ]:
configs = [("greedy", None, None)] + [
    ("beam", k, norm) for k in ([1, 3] if FAST_DEV else [1, 3, 5, 10]) for norm in (True,)
]
if not FAST_DEV:
    configs.append(("beam", 5, False))      # k=5 WITHOUT length normalisation

results = []
all_preds = {}
for decoder, k, norm in configs:
    label = "greedy" if decoder == "greedy" else f"beam k={k}" + ("" if norm else " (no len-norm)")
    print(f"\n--- {label} ---")
    bleu, preds = evaluate(model, eval_keys, mapping, features, tokenizer, idx_to_word,
                           max_length, decoder=decoder,
                           beam_width=k or 1, length_norm=bool(norm))
    results.append((label, bleu))
    all_preds[label] = preds

# Print as a markdown table, ready to paste into the report
print("\n| Decoder | BLEU-1 | BLEU-2 | BLEU-3 | BLEU-4 | Time (s) |")
print("|---|---|---|---|---|---|")
for label, b in results:
    print(f"| {label} | {b['BLEU-1']:.4f} | {b['BLEU-2']:.4f} | "
          f"{b['BLEU-3']:.4f} | {b['BLEU-4']:.4f} | {b['seconds']:.0f} |")

with open(os.path.join(WORKING_DIR, "bleu_results.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["decoder", "BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4", "seconds"])
    for label, b in results:
        w.writerow([label] + [f"{b[c]:.4f}" for c in
                              ("BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4")] + [f"{b['seconds']:.1f}"])
print("\nwrote bleu_results.csv")

In [ ]:
# Grouped bar chart of the comparison - a compact report figure.
labels = [lab for lab, _ in results]
x = np.arange(len(labels))
width = 0.2
plt.figure(figsize=(9, 4))
for i, n in enumerate((1, 2, 3, 4)):
    plt.bar(x + (i - 1.5) * width, [b[f"BLEU-{n}"] for _, b in results],
            width, label=f"BLEU-{n}")
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylabel("score")
plt.title("Decoding strategy comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "decoder_comparison.png"), dpi=150)
plt.show()

## Step 13 — Error analysis

The brief asks for an error analysis identifying weaknesses and proposing improvements.
A single mean BLEU hides almost everything, so this looks at four things:

1. **Score distribution** — is the mean dragged by a few total failures, or uniformly poor?
2. **Score vs reference length** — does the model fail specifically on long captions?
3. **Caption diversity** — how often does it fall back on the same generic caption?
4. **Worst cases** — the actual failures, read side by side with the references.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# Best available predictions: prefer the top beam config, else greedy
best_label = max(results, key=lambda r: r[1]["BLEU-4"])[0]
preds = all_preds[best_label]
print(f"analysing: {best_label}\n")

smooth = SmoothingFunction().method1
per_image = []
for key, pred in zip(eval_keys, preds):
    refs = [strip_markers(c) for c in mapping[key]]
    s = sentence_bleu(refs, pred, weights=(0.25,) * 4, smoothing_function=smooth)
    per_image.append({"key": key, "pred": " ".join(pred),
                      "bleu4": s, "ref_len": np.mean([len(r) for r in refs]),
                      "pred_len": len(pred)})

scores = np.array([p["bleu4"] for p in per_image])
print(f"BLEU-4  mean={scores.mean():.4f}  median={np.median(scores):.4f}")
print(f"        min ={scores.min():.4f}  max   ={scores.max():.4f}")
print(f"captions scoring < 0.05: {(scores < 0.05).sum()} / {len(scores)}")

# 1. distribution + 2. score vs reference length
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(scores, bins=20, edgecolor="black")
ax[0].axvline(scores.mean(), color="red", ls="--", label=f"mean {scores.mean():.3f}")
ax[0].set_xlabel("sentence BLEU-4"); ax[0].set_ylabel("images")
ax[0].set_title("Score distribution"); ax[0].legend()

ax[1].scatter([p["ref_len"] for p in per_image], scores, alpha=0.5)
ax[1].set_xlabel("mean reference length (words)"); ax[1].set_ylabel("sentence BLEU-4")
ax[1].set_title("Score vs caption length")
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "error_analysis.png"), dpi=150)
plt.show()

In [ ]:
# 3. Diversity - generic-caption collapse is the classic failure of this architecture
from collections import Counter
counts = Counter(p["pred"] for p in per_image)
unique = len(counts)
print(f"unique captions : {unique} / {len(per_image)}  ({unique / len(per_image):.1%})")
print("\nmost repeated predictions:")
for cap, c in counts.most_common(5):
    print(f"  {c:>3}x  {cap}")

# Repeated-token check (a known LSTM decoding pathology)
repeats = [p for p in per_image
           if any(c > 2 for c in Counter(p["pred"].split()).values())]
print(f"\ncaptions with a word repeated 3+ times: {len(repeats)}")
for p in repeats[:3]:
    print("  ", p["pred"])

In [ ]:
# 4. Worst cases, read against the references - the qualitative core of the analysis
print("=" * 78)
print("WORST PREDICTIONS")
print("=" * 78)
for p in sorted(per_image, key=lambda d: d["bleu4"])[:5]:
    print(f"\n[{p['key']}]  BLEU-4 = {p['bleu4']:.4f}")
    print(f"  generated : {p['pred']}")
    print("  references:")
    for c in mapping[p["key"]][:3]:
        print("    ", " ".join(strip_markers(c)))

print("\n" + "=" * 78)
print("BEST PREDICTIONS")
print("=" * 78)
for p in sorted(per_image, key=lambda d: -d["bleu4"])[:3]:
    print(f"\n[{p['key']}]  BLEU-4 = {p['bleu4']:.4f}")
    print(f"  generated : {p['pred']}")
    print(f"  reference : {' '.join(strip_markers(mapping[p['key']][0]))}")

### Qualitative examples with the images

Captions next to the actual photographs. Pick a few of these for the report — they
communicate the model's behaviour far more efficiently than prose, and figures cost almost
nothing against the 2,000-word limit.

**Failure modes to look for and name explicitly:**

- **Generic captions** — "a man in a red shirt" fits thousands of images; the model learned
  the caption prior rather than the image
- **Colour and count errors** — "two dogs" for one dog; these come from the flat 4,096-d
  feature having no spatial structure
- **Hallucinated objects** — plausible co-occurrences that are not in the image
- **Repeated tokens** — a known LSTM decoding pathology

In [ ]:
from tensorflow.keras.preprocessing.image import load_img

sample = per_image[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.ravel(), sample):
    ax.imshow(load_img(os.path.join(BASE_DIR, "Images", p["key"] + ".jpg"),
                       target_size=(224, 224)))
    ax.axis("off")
    ref = " ".join(strip_markers(mapping[p["key"]][0]))
    ax.set_title(f"BLEU-4 {p['bleu4']:.3f}\nGEN: {p['pred'][:60]}\nREF: {ref[:60]}",
                 fontsize=8, loc="left")
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "caption_examples.png"), dpi=150)
plt.show()

## Next steps

Once this notebook runs clean with `FAST_DEV = True`:

1. **Set `FAST_DEV = False`** and run the whole notebook top to bottom (~1 h training,
   ~30 min evaluation). Those are the numbers for the report.
2. **Hyperparameter tuning** — re-run with `LSTM_UNITS = 512`, then `DROPOUT = 0.2`,
   changing one variable at a time against this baseline. Record BLEU-4 and final
   validation loss for each.
3. **Write up** against the 2,000-word budget in `PLAN.md`.
4. **Rehearse the Q&A** using the question bank in `PLAN.md` Section 7 — it is 50% of the
   mark, and larger than every other component combined.

### Artefacts this notebook writes

| File | Purpose |
|---|---|
| `model.keras` | Trained model |
| `tokenizer.pkl` | Word index — decoding needs the exact same mapping |
| `history.csv` | Per-epoch train/validation loss |
| `loss_curve.png` | Report figure |
| `model.png` | Architecture diagram |
| `bleu_results.csv` | The decoder comparison numbers |
| `decoder_comparison.png` | Report figure |
| `error_analysis.png` | Report figure |
| `caption_examples.png` | Report figure |

Add these to `.gitignore` rather than committing them — `model.keras` is large, and the
figures are regenerated by a single run.